# 04 — `loss.backward()` olmadan eğitim


In [ ]:
import random
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3

def build_dataset(items):
    X, Y = [], []
    for word in items:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(Xtr.shape, Xdev.shape, Xte.shape, vocab_size)

In [ ]:
n_embd, n_hidden, batch_size = 10, 200, 32
n = batch_size
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
lossi = []

with torch.no_grad():
    for step in range(20000):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix]

        emb = C[Xb]
        embcat = emb.view(batch_size, -1)
        hprebn = embcat @ W1 + b1
        bnmean = hprebn.mean(0, keepdim=True)
        bnvar = hprebn.var(0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5) ** -0.5
        bnraw = (hprebn - bnmean) * bnvar_inv
        hpreact = bngain * bnraw + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Yb)

        dlogits = F.softmax(logits, dim=1)
        dlogits[range(n), Yb] -= 1
        dlogits /= n
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)
        dhpreact = (1 - h ** 2) * dh
        dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)
        dhprebn = bngain * bnvar_inv / n * (
            n * dhpreact
            - dhpreact.sum(0, keepdim=True)
            - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0, keepdim=True)
        )
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)
        demb = dembcat.view_as(emb)
        dC = torch.zeros_like(C)
        dC.index_add_(0, Xb.reshape(-1), demb.reshape(-1, n_embd))
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

        lr = 0.1 if step < 15000 else 0.01
        for parameter, grad in zip(parameters, grads):
            parameter += -lr * grad
        lossi.append(loss.item())
        if step % 5000 == 0 or step == 19999:
            print(f"{step:5d}: {loss.item():.4f}")

assert sum(lossi[-100:]) / 100 < sum(lossi[:100]) / 100

In [ ]:
with torch.no_grad():
    emb = C[Xtr].view(Xtr.shape[0], -1)
    hprebn_all = emb @ W1 + b1
    bnmean = hprebn_all.mean(0, keepdim=True)
    bnvar = hprebn_all.var(0, keepdim=True, unbiased=True)

    def split_loss(X, Y):
        emb = C[X].view(X.shape[0], -1)
        hprebn = emb @ W1 + b1
        hpreact = bngain * (hprebn - bnmean) * (bnvar + 1e-5) ** -0.5 + bnbias
        logits = torch.tanh(hpreact) @ W2 + b2
        return F.cross_entropy(logits, Y).item()

    print(f"train loss: {split_loss(Xtr, Ytr):.4f}")
    print(f"validation loss: {split_loss(Xdev, Ydev):.4f}")
    print(f"test loss: {split_loss(Xte, Yte):.4f}")